# 03 — Hybrid Search with HelixIndex

`HelixIndex` combines keyword search (`RelevanceIndex`) and vector search (`SimlarEngine`) and merges their rankings with Reciprocal Rank Fusion.

In this notebook we index 500 real Quora questions and compare text-only, vector-only, and hybrid results side by side.

**What we cover**
- Attaching an `embedder` so documents and queries are embedded for you
- Searching with text alone, a vector alone, or both
- Letting the engine tune how deep each side searches, with `speed_preference`
- Saving and reloading the index

In [1]:
%pip install -q datasets sentence-transformers simlar

Note: you may need to restart the kernel to use updated packages.


## Load the dataset and the embedding model

We use the corpus split of [`BeIR/quora`](https://huggingface.co/datasets/BeIR/quora) — real questions from Quora — and `all-MiniLM-L6-v2` (384-dim, ~80 MB download once) as the embedding model.

In [2]:
import tempfile
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from simlar import HelixIndex, load_from_directory

ds = load_dataset("BeIR/quora", "corpus", split="corpus[:500]")
corpus = ds["text"]
ids    = ds["_id"]

model = SentenceTransformer("all-MiniLM-L6-v2")

id_to_text = dict(zip(ids, corpus))

print(f"Loaded {len(corpus)} questions")
print(f"Sample: {corpus[0][:100]}")

/home/aramirez/anaconda3/envs/simlar/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14321.67it/s]

Loaded 500 questions
Sample: What is the step by step guide to invest in share market in india?


## Build the index

With an `embedder` attached, `add()` takes text alone: the texts go to `RelevanceIndex`, and their embeddings, computed by the embedder, go to `SimlarEngine`. If you already have vectors, pass them as `vectors=` and the embedder is skipped.

We don't set `text_k` or `vector_k` (how many candidates each side contributes to the fusion). The engine tunes them to the corpus size and `top_k`.

In [3]:
index = HelixIndex(top_k=5, embedder=model.encode)
index.add(ids=list(ids), texts=list(corpus))

print(f"Index size: {index.size}")

Index size: 500


## Search

`search()` with text alone embeds the query and runs both sides. Pass `query_vector` alone for a vector-only search, or query the text index directly for keyword-only results.

In [4]:
QUERY_TEXT = "What is the best programming language to learn first?"
print(f"Query: '{QUERY_TEXT}'\n")

def show(results):
    for r in results:
        print(f"  rank={r.rank}  score={r.score:.4f}  |  {id_to_text[r.id][:80]}")
    print()

print("Text only:")
show(index.text_index.search(QUERY_TEXT, k=5))

print("Vector only:")
show(index.search(query_vector=model.encode([QUERY_TEXT]), k=5))

print("Hybrid (text + vector):")
show(index.search(QUERY_TEXT, k=5))

Query: 'What is the best programming language to learn first?'

Text only:


  rank=0  score=5.7289  |  What is Java programming? How To Learn Java Programming Language ?
  rank=1  score=3.7568  |  How do I learn a computer language like java?
  rank=2  score=2.8123  |  What are the best ways to learn French?
  rank=3  score=2.5730  |  What's the best way to start learning robotics?
  rank=4  score=2.3729  |  What was your first sexual experience?

Vector only:
  rank=0  score=0.3333  |  How do I learn a computer language like java?
  rank=1  score=0.2500  |  What is Java programming? How To Learn Java Programming Language ?
  rank=2  score=0.2000  |  What's the best way to start learning robotics?
  rank=3  score=0.1667  |  What is the best way to start robotics? Which is the best development board that
  rank=4  score=0.1429  |  Which test series is the best for GATE computer science stream?

Hybrid (text + vector):
  rank=0  score=0.5833  |  How do I learn a computer language like java?
  rank=1  score=0.5833  |  What is Java programming? How To Learn Java P

A list of queries is a batch: you get one result list per query, and all queries are embedded in one call.

In [5]:
QUERIES = [
    "How do I invest in the stock market?",
    "How can I learn to speak Spanish quickly?",
]
for query, results in zip(QUERIES, index.search(QUERIES, k=3)):
    print(f"Query: '{query}'")
    show(results)

Query: 'How do I invest in the stock market?'
  rank=0  score=0.6667  |  What is the step by step guide to invest in share market?
  rank=1  score=0.5000  |  What is the step by step guide to invest in share market in india?
  rank=2  score=0.4000  |  How can I best invest ₹5000 over the next 6 months?

Query: 'What are the best ways to lose weight?'
  rank=0  score=0.3768  |  At what age, how, and where did you lose your virginity?
  rank=1  score=0.3733  |  What is the best diet for a growing decathlete?
  rank=2  score=0.3095  |  What's the best way to jump rope?



## Speed vs. recall

`speed_preference` trades latency for recall when the engine picks the search depth: `"fastest"`, `"fast"`, `"balanced"` (default), `"accurate"` or `"most accurate"`. `expected_recall` reports the recall the tuning model expects, or `None` when the corpus is too small for it to estimate (as with these 500 questions). The tuned depth is recomputed whenever the corpus changes.

In [6]:
fast = HelixIndex(top_k=5, embedder=model.encode, speed_preference="fastest")
fast.add(ids=list(ids), texts=list(corpus))

print(f"speed_preference={fast.speed_preference}  expected_recall={fast.expected_recall}\n")
show(fast.search(QUERY_TEXT, k=5))

speed_preference=fastest  expected_recall=None

  rank=0  score=0.5833  |  How do I learn a computer language like java?
  rank=1  score=0.5833  |  What is Java programming? How To Learn Java Programming Language ?
  rank=2  score=0.3667  |  What's the best way to start learning robotics?
  rank=3  score=0.2833  |  What are the best ways to learn French?
  rank=4  score=0.2250  |  How can I learn computer security?



## Save and load

The index (including `speed_preference`) is saved to disk. Embedders are not saved, so attach one again when loading.

In [7]:
with tempfile.TemporaryDirectory() as tmp:
    index.save(tmp)
    index2 = load_from_directory(tmp, embedder=model.encode)
    print(f"Loaded index size: {index2.size}\n")
    show(index2.search(QUERY_TEXT, k=5))

Loaded index size: 500

  rank=0  score=0.5833  |  How do I learn a computer language like java?
  rank=1  score=0.5833  |  What is Java programming? How To Learn Java Programming Language ?
  rank=2  score=0.3667  |  What's the best way to start learning robotics?
  rank=3  score=0.2833  |  What are the best ways to learn French?
  rank=4  score=0.2250  |  How can I learn computer security?

